# Tool Calling

## Arxiv Tool Calling
langchain_arxiv does not work and has terrible documentation, so using the arxiv library directly instead.

The three main types in the `arxiv` library are as follows:
|Type|Purpose|
|---|---|
|arxiv.Client|Reusable fetcher — holds pagination/retry config and the connection pool. Call client.results(search).|
|arxiv.Search|Describes a query — query, id_list, max_results, sort_by, sort_order.|
|arxiv.Result|A single paper — has .title, .authors, .summary, .published, .pdf_url, .entry_id, etc.|

In [ ]:
import arxiv


### Basic Calling

In [ ]:

client = arxiv.Client()
search = arxiv.Search(query="Machine Learning", max_results=2, sort_by=arxiv.SortCriterion.Relevance)

results = client.results(search)

for result in results:
    print(result.title)
    print(result.summary)
    print(result.authors)
    print(result.published)
    print(result.pdf_url)

### Advance query syntax

The `arxiv` library supports advanced query syntax for more precise searches. You can combine author, title, abstract, and other fields using logical operators like `AND`, `OR`, and `NOT`.

Examples:
- `au:del_maestro AND ti:checkerboard` — papers authored by Del Maestro with "checkerboard" in the title.
- `ti:"machine learning" AND abs:"neural networks"` — papers with "machine learning" in the title and "neural networks" in the abstract.
- `au:smith NOT ti:quantum` — papers authored by Smith but not having "quantum" in the title.
- `abs:"deep learning" OR abs:"neural networks"` — papers with "deep learning" or "neural networks" in the abstract.

In [ ]:
search = arxiv.Search(query="au:del_maestro AND ti:checkerboard")
first = next(client.results(search))
print(first.title)
print(first.summary)
print(first.authors)
print(first.published)
print(first.pdf_url)


In [ ]:

# from langchain_core.tools import tool
# import arxiv

# @tool
# def arxiv_search(query: str) -> str:
#     """Search arXiv for papers matching the query. Returns titles + summaries."""
#     client = arxiv.Client()
#     search = arxiv.Search(query=query, max_results=2, sort_by=arxiv.SortCriterion.Relevance)
#     docs = []
#     for r in client.results(search):
#         docs.append(f"Title: {r.title}\nPublished: {r.published.date()}\nSummary: {r.summary[:500]}")
#     return "\n\n---\n\n".join(docs) if docs else "No results found."

# # Now this works with .invoke() and agents:
# arxiv_search.invoke("machine learning")

## Wikipedia Tool Calling

### X basic Calling with deprecated langchain libs

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper_wiki = WikipediaAPIWrapper(top_k=5, document_content_char_limit=1000)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
result = wiki.invoke("Artificial Intelligence")
print(result)

### Using latest Wikipedia API (wikipedia-api) from PyPI

In [ ]:
import wikipediaapi

Key differences between the sync and async API:

- `summary`, `text`, `sections`, `langlinks`, `links`, `backlinks`, `categories`, `categorymembers`, `coordinates`, `images`, `pageid`, `fullurl`, `displaytitle`, … are explicit @property definitions in both APIs. 
In the async API every such property returns a coroutine: await page.summary, await page.sections, await page.pageid, etc.

- `title`, `ns`, `namespace`, `language`, `variant` are plain @property values in both APIs (no await needed).

- `exists()` is a plain method in the sync API; a coroutine method in the async API: await page.exists().

- `section_by_title()` and `sections_by_title()` are plain synchronous methods in both APIs.

In [ ]:
# Synchronous client
wiki = wikipediaapi.Wikipedia(user_agent='MyProjectName (merlin@example.com)', language='en')

page_py = wiki.page('Python_(programming_language)')
print("Page - Title: %s" % page_py.title)
# print("Page - Summary: %s" % page_py.summary[0:60])
print(page_py.summary)
print("Page - URL: %s" % page_py.fullurl)

In [ ]:
wiki = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

p_wiki = wiki.page("Test 1")
print(p_wiki.text)

In [ ]:
wiki_html = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.HTML
)
p_html = wiki_html.page("Test 1")
print(p_html.text)

In [ ]:
import asyncio

# Asynchronous client
wiki = wikipediaapi.AsyncWikipedia(user_agent='MyProjectName (merlin@example.com)', language='en')
page_py = wiki.page('Python_(programming_language)')
print('Article Summary:', await page_py.summary) # need to use await
print(page_py.title) # cannot use await 
print('Article URL:', await page_py.fullurl) # needs await


In [ ]:
wiki_wiki = wikipediaapi.AsyncWikipedia(
        user_agent='MyProjectName (merlin@example.com)',
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )
page = wiki_wiki.page("Test 1")
text = await page.text
print(text)

# Main Code

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv

Initialization of LLM

In [ ]:
load_dotenv()

# ollama_llm = ChatOllama(model="gemma4:e4b", api_key="OLLAMA_API_KEY")
lmstudio_llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    model="microsoft/phi-4-mini-reasoning", 
    api_key="lm_studio",
    )
# openrouter_llm = ChatOpenRouter(model="gpt-oss-20b", api_key="OPENROUTER_API_KEY")


Define the State

In [ ]:
# When implementing conversational story, need to use `add_message` method to add messages to the conversation 
# instead of replacing the entire message list.

from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    
def chat_node(state: ChatState):
    # take user input and add it to the messages list
    messages = state["messages"]
    # send it to the LLM and get the response
    response = lmstudio_llm.invoke(messages)
    # response stored in the messages list
    return {"messages": [response]}

In [ ]:
graph = StateGraph(ChatState)

# add  nodes
graph.add_node('chat_node', chat_node)

# add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

# compile the graph
chatbot = graph.compile()

In [ ]:
# input is given as list of messages, where each message is a dictionary with keys "type" and "content". The "type" can be either 
# "human" or "ai", and the "content" is the text of the message. 
# The initial state of the chatbot is defined as follows:
initial_state = {
    "messages": [HumanMessage(content = "Hello, do you know Rust programming?") ]
}

# chatbot.invoke(initial_state)
chatbot.invoke(initial_state)['messages'][-1].content

A while loop is used to continuously prompt the user for input, and the chatbot responds based on the current state. The state is updated with each interaction, allowing the chatbot to maintain context throughout the conversation.

In [ ]:
while True:
    user_message = input('Type Here: ')
    print('User: ', user_message)
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        print('Exiting...')
        break
    response = chatbot.invoke(
        {
            "messages": [HumanMessage(content=user_message)]
        }
    )

    print('AI: ', response['messages'][-1].content)